In [ ]:
# Trials of LLM-as-a-judge evaluation for errors

In [ ]:
#TSA on document-level with GPT-o4-mini
# ----------------------------
# CONFIGURATION
# ----------------------------
TEXT_FOLDER = "Texts"
OUTPUT_FOLDER = "Annotations"
MODEL = "o4-mini-2025-04-16"   

client = AsyncOpenAI()

# ----------------------------
# PROMPT TEMPLATE
# ----------------------------
PROMPT_TEMPLATE = """You are a careful and balanced annotator for machine translation quality.
Your task is to identify translation errors with appropriate confidence.

## Error Categories:
- Accuracy: addition, mistranslation, omission, untranslated text
- Fluency: character encoding, grammar, inconsistency, punctuation, register, spelling,
- Style: awkward phrasing
- Terminology: inappropriate for context, inconsistent use
- Other: non-translation, other issues

## Severity Classification:
- Critical: Errors that impact meaning and usability greatly. They  may misrepresent or damage the reputation of the author or publishing house, causes the text to stop working as a literary artefact and affects the communicative flow or if the language is perceived as offensive (when unintended).
- Major: Errors that may confuse or mislead the reader or hinder the understanding of the text due to significant change in meaning or because errors appear in a visible or important part of the content
- Minor: Errors that don't lead to loss of meaning and wouldn't confuse or mislead the reader but would be noticed, would decrease stylistic quality, fluency or clarity, or would make the content less appealing

## CRITICAL OUTPUT REQUIREMENTS:
- Mark errors only when you have clear evidence they are incorrect
- Wrap error spans in the translation with tags: <v0>, <v1>, <v2>, etc.
- Use tag numbers in sequential order starting from <v0>. Do not skip numbers or use them out of order.
- For omissions, place empty tags <vN></vN> where the missing text should have been.
- Return JSON with "annotated_translation" and "errors"
- NO comments, explanations, or additional text

### SOURCE TEXT:
{source}

### TRANSLATION:
{translation}
"""

# ----------------------------
# Helper: call the model
# ----------------------------


async def annotate(source, translation):
    prompt = PROMPT_TEMPLATE.format(source=source, translation=translation)

    response = await client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        # temperature removed for reasoning models
    )

    # Correct access to the content
    return response.choices[0].message.content
# ----------------------------
# Parsing filenames to find pairs
# ----------------------------
def parse_filename(filename):
    # Example: EN-NL_Poem_HT.txt
    name, _ = os.path.splitext(filename)
    try:
        st_tt, genre, modality = name.split("_")
        st, tt = st_tt.split("-")
        return st, tt, genre, modality
    except ValueError:
        return None


# ----------------------------
# Build index of available files
# ----------------------------
def index_texts():
    index = {}
    for fname in os.listdir(TEXT_FOLDER):
        parsed = parse_filename(fname)
        if not parsed:
            continue

        st, tt, genre, mod = parsed
        key = (st, tt, genre)

        if key not in index:
            index[key] = {}

        path = os.path.join(TEXT_FOLDER, fname)
        index[key][mod] = path

    return index


# ----------------------------
# MAIN WORKFLOW
# ----------------------------
async def main():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    text_index = index_texts()

    for (st, tt, genre), files in text_index.items():
        if "ST" not in files:
            continue  # cannot annotate without source

        # load source
        with open(files["ST"], "r", encoding="utf-8") as f:
            source_text = f.read()

        for modality in ["HT", "MT", "PE"]:
            if modality not in files:
                continue

            with open(files[modality], "r", encoding="utf-8") as f:
                translation_text = f.read()

            print(f"Annotating {st}-{tt} {genre} {modality}...")

            annotation_json = await annotate(source_text, translation_text)

            # save output
            out_name = f"{st}-{tt}_{genre}_{modality}_ANNOTATED.json"
            out_path = os.path.join(OUTPUT_FOLDER, out_name)

            with open(out_path, "w", encoding="utf-8") as f:
                f.write(annotation_json)

    print("Done! Annotations stored in folder:", OUTPUT_FOLDER)


# run async
if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
#AutoMQM model
import os
import re
import openai


client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# --- CONFIGURATION ---
TEXTS_FOLDER = "Texts"
OUTPUT_FOLDER = "Annotations"

# Source-target modalities to annotate
MODALITIES = ["HT", "MT", "PE"]

# Supported languages, genres, modalities
ST_OPTIONS = ["EN", "RU"]
TT_OPTIONS = ["NL", "CA"]
GENRES = ["Poem", "ShortStory", "Thriller"]

# Make output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Prompt template
PROMPT_TEMPLATE = """
Based on the source segment and machine translation surrounded with triple backticks, identify error types in the translation and classify them. The categories of errors are: accuracy
(addition, mistranslation, omission, untranslated text), fluency (character encoding, grammar, inconsistency, punctuation, register, spelling), style (awkward), terminology (inappropriate for context, inconsistent use), non-translation, other, or no-error.
Each error is classified as one of three categories: critical, major, and minor. Critical errors impact meaning and usability greatly. They may misrepresent or damage the reputation of the author or publishing house, causes the text to stop working as a literary artefact and affects the communicative flow or if the language is perceived as offensive (when unintended). Major errors may confuse or mislead the reader or hinder the understanding of the text due to significant change in meaning or because errors appear in a visible or important part of the content, whereas minor errors are errors that don't lead to loss of meaning and wouldn't confuse or mislead the reader but would be noticed, would decrease stylistic quality, fluency or clarity, or would make the content less appealing.

Source:
{source_text}

Translation:
{translation_text}

"""

# --- FUNCTION TO CALL GPT-4 ---
def annotate_translation(source_text, translation_text):
    prompt = PROMPT_TEMPLATE.format(source_text=source_text, translation_text=translation_text)
    response = openai.chat.completions.create(
        model="o4-mini-2025-04-16",
        messages=[
            {"role": "system", "content": "You are a reasoning model that annotates translations."},
            {"role": "user", "content": prompt}
        ],
        
    )
    return response.choices[0].message.content


# --- FUNCTION TO READ FILE CONTENT ---
def read_text_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

# --- MAIN PROCESS ---
# Pattern: ST-TT_genre_modality
pattern = re.compile(r"([A-Z]{2})-([A-Z]{2})_([A-Za-z]+)_([A-Z]{2})\.txt")

# Index files
files_index = {}
for filename in os.listdir(TEXTS_FOLDER):
    match = pattern.match(filename)
    if match:
        st, tt, genre, modality = match.groups()
        key = f"{st}-{tt}_{genre}"
        files_index.setdefault(key, {})[modality] = filename

# Annotate translations
for key, modalities_dict in files_index.items():
    if "ST" not in modalities_dict:  # Skip if source file missing
        continue

    source_file = modalities_dict["ST"]
    source_text = read_text_file(os.path.join(TEXTS_FOLDER, source_file))

    for mod in MODALITIES:
        if mod in modalities_dict:
            translation_file = modalities_dict[mod]
            translation_text = read_text_file(os.path.join(TEXTS_FOLDER, translation_file))

            print(f"Annotating {translation_file}...")
            annotation = annotate_translation(source_text, translation_text)

            # Save output
            out_filename = f"{os.path.splitext(translation_file)[0]}_annotation.txt"
            with open(os.path.join(OUTPUT_FOLDER, out_filename), "w", encoding="utf-8") as f_out:
                f_out.write(annotation)

print("Annotation complete!")



In [ ]:
#Gemba-MQM model
import os
import re
import openai

# Set your OpenAI API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

TEXTS_FOLDER = "Texts"

few_shots = {
    "ende": {
        "source_lang": "English",
        "source_seg": "I do apologise about this, we must gain permission from the account holder to discuss an order with another person, I apologise if this was done previously, however, I would not be able to discuss this with yourself without the account holders permission.",
        "target_lang": "German",
        "target_seg": "Ich entschuldige mich dafür, wir müssen die Erlaubnis einholen, um eine Bestellung mit einer anderen Person zu besprechen. Ich entschuldige mich, falls dies zuvor geschehen wäre, aber ohne die Erlaubnis des Kontoinhabers wäre ich nicht in der Lage, dies mit dir involvement.",
        "answer": """Critical:
no-error
Major:
accuracy/mistranslation - "involvement"
accuracy/omission - "the account holder"
Minor:
fluency/grammar - "wäre"
fluency/register - "dir"
""",
    },
    "encs": {
        "source_lang": "English",
        "source_seg": "Talks have resumed in Vienna to try to revive the nuclear pact, with both sides trying to gauge the prospects of success after the latest exchanges in the stop-start negotiations.",
        "target_lang": "Czech",
        "target_seg": "Ve Vídni se ve Vídni obnovily rozhovory o oživení jaderného paktu, přičemž obě partaje se snaží posoudit vyhlídky na úspěch po posledních výměnách v jednáních.",
        "answer": """Critical:
no-error
Major:
accuracy/addition - "ve Vídni"
accuracy/omission - "the stop-start"
Minor:
terminology/inappropriate for context - "partaje"
""",
    },
    "zhen": {
        "source_lang": "Chinese",
        "source_seg": "大众点评乌鲁木齐家居卖场频道为您提供高铁居然之家地址，电话，营业时间等最新商户信息，找装修公司，就上大众点评",
        "target_lang": "English",
        "target_seg": "Urumqi Home Furnishing Store Channel provides you with the latest business information such as the address, telephone number, business hours, etc., of high-speed rail, and find a decoration company, and go to the reviews.",
        "answer": """Critical:
accuracy/addition - "of high-speed rail"
Major:
accuracy/mistranslation - "go to the reviews"
Minor:
style/awkward - "etc.,"
""",
    },
}

pattern = r"(?P<ST>EN|RU)-(?P<TT>NL|CA)_(?P<genre>Poem|ShortStory|Thriller)_(?P<modality>HT|MT|PE|ST)\.txt"
MODALITIES_TO_ANNOTATE = ["HT", "MT", "PE"]

def get_file_dict():
    file_dict = {}
    for filename in os.listdir(TEXTS_FOLDER):
        match = re.match(pattern, filename)
        if match:
            groups = match.groupdict()
            key = (groups["ST"], groups["TT"], groups["genre"])
            if key not in file_dict:
                file_dict[key] = {}
            file_dict[key][groups["modality"]] = filename
    return file_dict

def read_file(filename):
    with open(os.path.join(TEXTS_FOLDER, filename), "r", encoding="utf-8") as f:
        return f.read()

def build_prompt(source_lang, source_seg, target_lang, target_seg):
    prompt = f"""
You are an annotator for the quality of machine translation. Your task is to identify errors and assess the quality of the translation.

{source_lang} source:
```{source_seg}```
{target_lang} translation:
```{target_seg}```

Based on the source segment and machine translation surrounded with triple backticks, identify error types in the translation and classify them. The categories of errors are: accuracy
(addition, mistranslation, omission, untranslated text), fluency (character encoding, grammar, inconsistency, punctuation, register, spelling), style (awkward), terminology (inappropriate for context, inconsistent use), non-translation, other, or no-error.
Each error is classified as one of three categories: critical, major, and minor. Critical errors impact meaning and usability greatly. They may misrepresent or damage the reputation of the author or publishing house, causes the text to stop working as a literary artefact and affects the communicative flow or if the language is perceived as offensive (when unintended). Major errors may confuse or mislead the reader or hinder the understanding of the text due to significant change in meaning or because errors appear in a visible or important part of the content, whereas minor errors are errors that don't lead to loss of meaning and wouldn't confuse or mislead the reader but would be noticed, would decrease stylistic quality, fluency or clarity, or would make the content less appealing.

Please follow the few-shot examples below:

"""
    for example in few_shots.values():
        prompt += f"{example['source_lang']} source:\n```{example['source_seg']}```\n"
        prompt += f"{example['target_lang']} translation:\n```{example['target_seg']}```\n"
        prompt += f"Answer:\n{example['answer']}\n\n"

    prompt += "Now annotate the following text:\n"
    return prompt

def annotate_text(source_seg, target_seg, source_lang, target_lang):
    prompt = build_prompt(source_lang, source_seg, target_lang, target_seg)
    response = openai.chat.completions.create(
        model="o4-mini-2025-04-16",
        messages=[
            {"role": "system", "content": "You are a professional translation annotator."},
            {"role": "user", "content": prompt}
        ],
    )
    return response.choices[0].message.content.strip()

def main():
    files = get_file_dict()
    for key, modalities in files.items():
        ST, TT, genre = key
        if "ST" not in modalities:
            continue
        source_seg = read_file(modalities["ST"])
        source_lang = "English" if ST == "EN" else "Russian"
        target_lang = "Dutch" if TT == "NL" else "Catalan"
        for modality in MODALITIES_TO_ANNOTATE:
            if modality in modalities:
                target_seg = read_file(modalities[modality])
                annotation = annotate_text(source_seg, target_seg, source_lang, target_lang)
                output_filename = f"{modalities[modality].replace('.txt','')}_annotation.txt"
                with open(os.path.join(TEXTS_FOLDER, output_filename), "w", encoding="utf-8") as f:
                    f.write(annotation)
                print(f"Annotated {modalities[modality]} -> {output_filename}")

if __name__ == "__main__":
    main()


In [ ]:
#Gemba-MQM model with numbered output

import os
import re
import openai

# Set your OpenAI API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

TEXTS_FOLDER = "Texts"

few_shots = {
    "ende": {
        "source_lang": "English",
        "source_seg": "I do apologise about this, we must gain permission from the account holder to discuss an order with another person, I apologise if this was done previously, however, I would not be able to discuss this with yourself without the account holders permission.",
        "target_lang": "German",
        "target_seg": "Ich entschuldige mich dafür, wir müssen die Erlaubnis einholen, um eine Bestellung mit einer anderen Person zu besprechen. Ich entschuldige mich, falls dies zuvor geschehen wäre, aber ohne die Erlaubnis des Kontoinhabers wäre ich nicht in der Lage, dies mit dir involvement.",
        "answer": """Critical:
no-error
Major:
1 "involvement" - accuracy/mistranslation 
2 "the account holder" - accuracy/omission 
Minor:
3 "wäre" - fluency/grammar 
4 "dir" - fluency/grammar 
""",
    },
    "encs": {
        "source_lang": "English",
        "source_seg": "Talks have resumed in Vienna to try to revive the nuclear pact, with both sides trying to gauge the prospects of success after the latest exchanges in the stop-start negotiations.",
        "target_lang": "Czech",
        "target_seg": "Ve Vídni se ve Vídni obnovily rozhovory o oživení jaderného paktu, přičemž obě partaje se snaží posoudit vyhlídky na úspěch po posledních výměnách v jednáních.",
        "answer": """Critical:
no-error
Major:
1 "ve Vídni" - fluency/grammar 
2 "the stop-start" - fluency/grammar
Minor:
3 "partaje" - terminology/inappropriate for context -
""",
    },
    "zhen": {
        "source_lang": "Chinese",
        "source_seg": "大众点评乌鲁木齐家居卖场频道为您提供高铁居然之家地址，电话，营业时间等最新商户信息，找装修公司，就上大众点评",
        "target_lang": "English",
        "target_seg": "Urumqi Home Furnishing Store Channel provides you with the latest business information such as the address, telephone number, business hours, etc., of high-speed rail, and find a decoration company, and go to the reviews.",
        "answer": """Critical:
1 "of high-speed rail" - accuracy/addition 
Major:
2 "go to the reviews" - accuracy/mistranslation 
Minor:
"etc.," - style/awkward 
""",
    },
}

pattern = r"(?P<ST>EN|RU)-(?P<TT>NL|CA)_(?P<genre>Poem|ShortStory|Thriller)_(?P<modality>HT|MT|PE|ST)\.txt"
MODALITIES_TO_ANNOTATE = ["HT", "MT", "PE"]

def get_file_dict():
    file_dict = {}
    for filename in os.listdir(TEXTS_FOLDER):
        match = re.match(pattern, filename)
        if match:
            groups = match.groupdict()
            key = (groups["ST"], groups["TT"], groups["genre"])
            if key not in file_dict:
                file_dict[key] = {}
            file_dict[key][groups["modality"]] = filename
    return file_dict

def read_file(filename):
    with open(os.path.join(TEXTS_FOLDER, filename), "r", encoding="utf-8") as f:
        return f.read()

def build_prompt(source_lang, source_seg, target_lang, target_seg):
    prompt = f"""
You are an annotator for the quality of machine translation. Your task is to identify errors and assess the quality of the translation.

{source_lang} source:
```{source_seg}```
{target_lang} translation:
```{target_seg}```

Based on the source segment and machine translation surrounded with triple backticks, identify error types in the translation and classify them. The categories of errors are: accuracy
(addition, mistranslation, omission, untranslated text), fluency (character encoding, grammar, inconsistency, punctuation, register, spelling), style (awkward), terminology (inappropriate for context, inconsistent use), non-translation, other, or no-error.
Each error is classified as one of three categories: critical, major, and minor. Critical errors impact meaning and usability greatly. They may misrepresent or damage the reputation of the author or publishing house, causes the text to stop working as a literary artefact and affects the communicative flow or if the language is perceived as offensive (when unintended). Major errors may confuse or mislead the reader or hinder the understanding of the text due to significant change in meaning or because errors appear in a visible or important part of the content, whereas minor errors are errors that don't lead to loss of meaning and wouldn't confuse or mislead the reader but would be noticed, would decrease stylistic quality, fluency or clarity, or would make the content less appealing.

Please follow the few-shot examples below:

"""
    for example in few_shots.values():
        prompt += f"{example['source_lang']} source:\n```{example['source_seg']}```\n"
        prompt += f"{example['target_lang']} translation:\n```{example['target_seg']}```\n"
        prompt += f"Answer:\n{example['answer']}\n\n"

    prompt += "Now annotate the following text:\n"
    return prompt

def annotate_text(source_seg, target_seg, source_lang, target_lang):
    prompt = build_prompt(source_lang, source_seg, target_lang, target_seg)
    response = openai.chat.completions.create(
        model="o4-mini-2025-04-16",
        messages=[
            {"role": "system", "content": "You are a professional translation annotator."},
            {"role": "user", "content": prompt}
        ],
    )
    return response.choices[0].message.content.strip()

def main():
    ANNOTATIONS_FOLDER = "Annotations"
    os.makedirs(ANNOTATIONS_FOLDER, exist_ok=True)  # Create folder if it doesn't exist

    files = get_file_dict()
    for key, modalities in files.items():
        ST, TT, genre = key
        if "ST" not in modalities:
            continue
        source_seg = read_file(modalities["ST"])
        source_lang = "English" if ST == "EN" else "Russian"
        target_lang = "Dutch" if TT == "NL" else "Catalan"

        for modality in MODALITIES_TO_ANNOTATE:
            if modality in modalities:
                target_seg = read_file(modalities[modality])
                annotation = annotate_text(source_seg, target_seg, source_lang, target_lang)

                output_filename = f"{modalities[modality].replace('.txt','')}_annotation.txt"
                output_path = os.path.join(ANNOTATIONS_FOLDER, output_filename)

                with open(output_path, "w", encoding="utf-8") as f:
                    f.write(annotation)

                print(f"Annotated {modalities[modality]} -> {output_path}")

if __name__ == "__main__":
    main()


In [ ]:
#Gemba-MQM model per sentence
import os
import re
import openai

# Set your OpenAI API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

TEXTS_FOLDER = "Texts"

few_shots = {
    "ende": {
        "source_lang": "English",
        "source_seg": "I do apologise about this, we must gain permission from the account holder to discuss an order with another person, I apologise if this was done previously, however, I would not be able to discuss this with yourself without the account holders permission.",
        "target_lang": "German",
        "target_seg": "Ich entschuldige mich dafür, wir müssen die Erlaubnis einholen, um eine Bestellung mit einer anderen Person zu besprechen. Ich entschuldige mich, falls dies zuvor geschehen wäre, aber ohne die Erlaubnis des Kontoinhabers wäre ich nicht in der Lage, dies mit dir involvement.",
        "answer": """Critical:
no-error
Major:
accuracy/mistranslation - "involvement"
accuracy/omission - "the account holder"
Minor:
fluency/grammar - "wäre"
fluency/register - "dir"
""",
    },
    "encs": {
        "source_lang": "English",
        "source_seg": "Talks have resumed in Vienna to try to revive the nuclear pact, with both sides trying to gauge the prospects of success after the latest exchanges in the stop-start negotiations.",
        "target_lang": "Czech",
        "target_seg": "Ve Vídni se ve Vídni obnovily rozhovory o oživení jaderného paktu, přičemž obě partaje se snaží posoudit vyhlídky na úspěch po posledních výměnách v jednáních.",
        "answer": """Critical:
no-error
Major:
accuracy/addition - "ve Vídni"
accuracy/omission - "the stop-start"
Minor:
terminology/inappropriate for context - "partaje"
""",
    },
    "zhen": {
        "source_lang": "Chinese",
        "source_seg": "大众点评乌鲁木齐家居卖场频道为您提供高铁居然之家地址，电话，营业时间等最新商户信息，找装修公司，就上大众点评",
        "target_lang": "English",
        "target_seg": "Urumqi Home Furnishing Store Channel provides you with the latest business information such as the address, telephone number, business hours, etc., of high-speed rail, and find a decoration company, and go to the reviews.",
        "answer": """Critical:
accuracy/addition - "of high-speed rail"
Major:
accuracy/mistranslation - "go to the reviews"
Minor:
style/awkward - "etc.,"
""",
    },
}

pattern = r"(?P<ST>EN|RU)-(?P<TT>NL|CA)_(?P<genre>Poem|ShortStory|Thriller)_(?P<modality>HT|MT|PE|ST)\.txt"
MODALITIES_TO_ANNOTATE = ["HT", "MT", "PE"]

def get_file_dict():
    file_dict = {}
    for filename in os.listdir(TEXTS_FOLDER):
        match = re.match(pattern, filename)
        if match:
            groups = match.groupdict()
            key = (groups["ST"], groups["TT"], groups["genre"])
            if key not in file_dict:
                file_dict[key] = {}
            file_dict[key][groups["modality"]] = filename
    return file_dict

def read_file(filename):
    with open(os.path.join(TEXTS_FOLDER, filename), "r", encoding="utf-8") as f:
        return f.read()

def build_prompt(source_lang, source_seg, target_lang, target_seg):
    prompt = f"""
You are an annotator for the quality of machine translation. Your task is to identify errors and assess the quality of the translation.

{source_lang} source:
```{source_seg}```
{target_lang} translation:
```{target_seg}```

Based on the source segment and machine translation surrounded with triple backticks, identify error types in the translation and classify them. The categories of errors are: accuracy
(addition, mistranslation, omission, untranslated text), fluency (character encoding, grammar, inconsistency, punctuation, register, spelling), style (awkward), terminology (inappropriate for context, inconsistent use), non-translation, other, or no-error.
Each error is classified as one of three categories: critical, major, and minor. Critical errors impact meaning and usability greatly. They may misrepresent or damage the reputation of the author or publishing house, causes the text to stop working as a literary artefact and affects the communicative flow or if the language is perceived as offensive (when unintended). Major errors may confuse or mislead the reader or hinder the understanding of the text due to significant change in meaning or because errors appear in a visible or important part of the content, whereas minor errors are errors that don't lead to loss of meaning and wouldn't confuse or mislead the reader but would be noticed, would decrease stylistic quality, fluency or clarity, or would make the content less appealing.

Please follow the few-shot examples below:

"""
    for example in few_shots.values():
        prompt += f"{example['source_lang']} source:\n```{example['source_seg']}```\n"
        prompt += f"{example['target_lang']} translation:\n```{example['target_seg']}```\n"
        prompt += f"Answer:\n{example['answer']}\n\n"

    prompt += "Now annotate the following text:\n"
    return prompt

def read_file_lines(filename):
    with open(os.path.join(TEXTS_FOLDER, filename), "r", encoding="utf-8") as f:
        return f.read().splitlines()

def annotate_text(source_seg, target_seg, source_lang, target_lang):
    prompt = build_prompt(source_lang, source_seg, target_lang, target_seg)
    response = openai.chat.completions.create(
        model="o4-mini-2025-04-16",
        messages=[
            {"role": "system", "content": "You are a professional translation annotator."},
            {"role": "user", "content": prompt}
        ],
    )
    return response.choices[0].message.content.strip()

def main():
    ANNOTATIONS_FOLDER = "Annotations"
    os.makedirs(ANNOTATIONS_FOLDER, exist_ok=True)

    files = get_file_dict()

    for key, modalities in files.items():
        ST, TT, genre = key
        if "ST" not in modalities:
            continue

        source_lang = "English" if ST == "EN" else "Russian"
        target_lang = "Dutch" if TT == "NL" else "Catalan"

        source_lines = read_file_lines(modalities["ST"])

        for modality in MODALITIES_TO_ANNOTATE:
            if modality not in modalities:
                continue

            target_lines = read_file_lines(modalities[modality])

            if len(source_lines) != len(target_lines):
                print(f"⚠️ Line mismatch in {modalities['ST']} vs {modalities[modality]}")
                min_len = min(len(source_lines), len(target_lines))
            else:
                min_len = len(source_lines)

            output_filename = f"{modalities[modality].replace('.txt','')}_annotation.json"
            output_path = os.path.join(ANNOTATIONS_FOLDER, output_filename)

            annotations_json = []

            for i in range(min_len):
                src = source_lines[i].strip()
                tgt = target_lines[i].strip()

                if not src and not tgt:
                    annotations_json.append({
                        "line": i + 1,
                        "source": src,
                        "target": tgt,
                        "annotation": "(no content)"
                    })
                    continue

                annotation = annotate_text(src, tgt, source_lang, target_lang)

                annotations_json.append({
                    "line": i + 1,
                    "source": src,
                    "target": tgt,
                    "annotation": annotation
                })

            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(annotations_json, f, ensure_ascii=False, indent=2)

            print(f"Annotated {modality} ({len(annotations_json)} lines) → {output_path}")


if __name__ == "__main__":
    main()


In [ ]:
#TSA model per sentence
import os
import re
import openai

# Set your OpenAI API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

TEXTS_FOLDER = "Texts"

few_shots = {
    "ende": {
        "source_lang": "English",
        "source_seg": "I do apologise about this, we must gain permission from the account holder to discuss an order with another person, I apologise if this was done previously, however, I would not be able to discuss this with yourself without the account holders permission.",
        "target_lang": "German",
        "target_seg": "Ich entschuldige mich dafür, wir müssen die Erlaubnis einholen, um eine Bestellung mit einer anderen Person zu besprechen. Ich entschuldige mich, falls dies zuvor geschehen wäre, aber ohne die Erlaubnis des Kontoinhabers wäre ich nicht in der Lage, dies mit dir involvement.",
        "answer": """Critical:
no-error
Major:
accuracy/mistranslation - "involvement"
accuracy/omission - "the account holder"
Minor:
fluency/grammar - "wäre"
fluency/register - "dir"
""",
    },
    "encs": {
        "source_lang": "English",
        "source_seg": "Talks have resumed in Vienna to try to revive the nuclear pact, with both sides trying to gauge the prospects of success after the latest exchanges in the stop-start negotiations.",
        "target_lang": "Czech",
        "target_seg": "Ve Vídni se ve Vídni obnovily rozhovory o oživení jaderného paktu, přičemž obě partaje se snaží posoudit vyhlídky na úspěch po posledních výměnách v jednáních.",
        "answer": """Critical:
no-error
Major:
accuracy/addition - "ve Vídni"
accuracy/omission - "the stop-start"
Minor:
terminology/inappropriate for context - "partaje"
""",
    },
    "zhen": {
        "source_lang": "Chinese",
        "source_seg": "大众点评乌鲁木齐家居卖场频道为您提供高铁居然之家地址，电话，营业时间等最新商户信息，找装修公司，就上大众点评",
        "target_lang": "English",
        "target_seg": "Urumqi Home Furnishing Store Channel provides you with the latest business information such as the address, telephone number, business hours, etc., of high-speed rail, and find a decoration company, and go to the reviews.",
        "answer": """Critical:
accuracy/addition - "of high-speed rail"
Major:
accuracy/mistranslation - "go to the reviews"
Minor:
style/awkward - "etc.,"
""",
    },
}

pattern = r"(?P<ST>EN|RU)-(?P<TT>NL|CA)_(?P<genre>Poem|ShortStory|Thriller)_(?P<modality>HT|MT|PE|ST)\.txt"
MODALITIES_TO_ANNOTATE = ["HT", "MT", "PE"]

def get_file_dict():
    file_dict = {}
    for filename in os.listdir(TEXTS_FOLDER):
        match = re.match(pattern, filename)
        if match:
            groups = match.groupdict()
            key = (groups["ST"], groups["TT"], groups["genre"])
            if key not in file_dict:
                file_dict[key] = {}
            file_dict[key][groups["modality"]] = filename
    return file_dict

def read_file(filename):
    with open(os.path.join(TEXTS_FOLDER, filename), "r", encoding="utf-8") as f:
        return f.read()

def build_prompt(source_lang, source_seg, target_lang, target_seg):
    prompt = f"""
You are an annotator for the quality of machine translation. Your task is to identify errors and assess the quality of the translation.

{source_lang} source:
```{source_seg}```
{target_lang} translation:
```{target_seg}```

Based on the source segment and machine translation surrounded with triple backticks, identify error types in the translation and classify them. The categories of errors are: accuracy
(addition, mistranslation, omission, untranslated text), fluency (character encoding, grammar, inconsistency, punctuation, register, spelling), style (awkward), terminology (inappropriate for context, inconsistent use), non-translation, other, or no-error.
Each error is classified as one of three categories: critical, major, and minor. Critical errors impact meaning and usability greatly. They may misrepresent or damage the reputation of the author or publishing house, causes the text to stop working as a literary artefact and affects the communicative flow or if the language is perceived as offensive (when unintended). Major errors may confuse or mislead the reader or hinder the understanding of the text due to significant change in meaning or because errors appear in a visible or important part of the content, whereas minor errors are errors that don't lead to loss of meaning and wouldn't confuse or mislead the reader but would be noticed, would decrease stylistic quality, fluency or clarity, or would make the content less appealing.

Please follow the few-shot examples below:

"""
    for example in few_shots.values():
        prompt += f"{example['source_lang']} source:\n```{example['source_seg']}```\n"
        prompt += f"{example['target_lang']} translation:\n```{example['target_seg']}```\n"
        prompt += f"Answer:\n{example['answer']}\n\n"

    prompt += "Now annotate the following text:\n"
    return prompt

def read_file_lines(filename):
    with open(os.path.join(TEXTS_FOLDER, filename), "r", encoding="utf-8") as f:
        return f.read().splitlines()

def annotate_text(source_seg, target_seg, source_lang, target_lang):
    prompt = build_prompt(source_lang, source_seg, target_lang, target_seg)
    response = openai.chat.completions.create(
        model="o4-mini-2025-04-16",
        messages=[
            {"role": "system", "content": "You are a professional translation annotator."},
            {"role": "user", "content": prompt}
        ],
    )
    return response.choices[0].message.content.strip()

def main():
    ANNOTATIONS_FOLDER = "Annotations"
    os.makedirs(ANNOTATIONS_FOLDER, exist_ok=True)

    files = get_file_dict()

    for key, modalities in files.items():
        ST, TT, genre = key
        if "ST" not in modalities:
            continue

        source_lang = "English" if ST == "EN" else "Russian"
        target_lang = "Dutch" if TT == "NL" else "Catalan"

        source_lines = read_file_lines(modalities["ST"])

        for modality in MODALITIES_TO_ANNOTATE:
            if modality not in modalities:
                continue

            target_lines = read_file_lines(modalities[modality])

            if len(source_lines) != len(target_lines):
                print(f"⚠️ Line mismatch in {modalities['ST']} vs {modalities[modality]}")
                min_len = min(len(source_lines), len(target_lines))
            else:
                min_len = len(source_lines)

            output_filename = f"{modalities[modality].replace('.txt','')}_annotation.json"
            output_path = os.path.join(ANNOTATIONS_FOLDER, output_filename)

            annotations_json = []

            for i in range(min_len):
                src = source_lines[i].strip()
                tgt = target_lines[i].strip()

                if not src and not tgt:
                    annotations_json.append({
                        "line": i + 1,
                        "source": src,
                        "target": tgt,
                        "annotation": "(no content)"
                    })
                    continue

                annotation = annotate_text(src, tgt, source_lang, target_lang)

                annotations_json.append({
                    "line": i + 1,
                    "source": src,
                    "target": tgt,
                    "annotation": annotation
                })

            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(annotations_json, f, ensure_ascii=False, indent=2)

            print(f"Annotated {modality} ({len(annotations_json)} lines) → {output_path}")


if __name__ == "__main__":
    main()
